<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Agents with Tools versus Tasks with Tools in CrewAI**


Estimated time needed: **30** minutes



Build a specialized agentic AI chatbot with CrewAI in only 30 minutes! This lab will teach you how to use tools with agentic workflows to solve real-world challenges. You'll create and orchestrate AI agents, define multi-step workflows, and explore the differences between assigning tools at the Agent level versus the Task level. This hands-on project is a necessity for all software and machine learning engineers looking to build more efficient and reliable multi-agent systems.

Here's your challenge: You've been hired by "The Daily Dish," a popular restaurant with a customer service problem. Chef Maria, the owner, needs your help: "My team spends hours answering the same questions about reservations, menu details, and our location," she explains. "Build me a chatbot that can handle these inquiries intelligently, so my staff can focus on creating exceptional dining experiences." Chef Maria provides a PDF of frequently asked questions (FAQs) to serve as the primary knowledge base.

You'll solve this by building a crew of AI agents that can search the FAQ document, browse the web for supplementary information, and synthesize the findings into a friendly, helpful response for the customer. Most importantly, you will learn how to make this process more robust and efficient by assigning tools directly to the tasks that need them.



## __Table of Contents__<a id="toc"></a>
<ol>
    <li><a href="#Learning-Objectives">Learning Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
            <li><a href="#Importing-Required-Libraries">Importing Required Libraries</a></li>
        </ol>
    </li>
    <li>
        <a href="#Project-Roadmap">Project Roadmap</a>
        <ol>
            <li><a href="#Agentic-AI-with-CrewAI">Agentic AI with CrewAI</a></li>
            <li><a href="#The-Role-of-Tools:-Agent-Centric-versus-Task-Centric">The Role of Tools: Agent-Centric versus Task-Centric</a></li>
        </ol>
    </li>
    <li><a href="#Writing-the-Code">Writing the Code</a>
        <ol>
            <li><a href="#Configuring-our-LLM">Configuring our LLM</a></li>
            <li><a href="#Defining-Tools-for-Information-Gathering">Defining Tools for Information Gathering</a></li>
            <li><a href="#Approach-1:-The-Standard-Method-(Agent-Centric-Tools)">Approach 1: The Standard Method (Agent-Centric Tools)</a></li>
            <li><a href="#Approach-2:-A-More-Focused-Method-(Task-Centric-Tools)">Approach 2: A More Focused Method (Task-Centric Tools)</a></li>
            <li><a href="#Chatbot-Execution">Chatbot Execution</a></li>
        </ol>
    </li>
    <li><a href="#Conclusion">Conclusion</a></li>
    <li><a href="#Extending-CrewAI-with-Custom-Functions">Extending CrewAI with Custom Functions</a></li>
</ol>


## Learning Objectives

After completing this lab you will be able to:

- Build a customer service chatbot using a multi-agent CrewAI workflow.
- Implement tools for information retrieval from both local documents (PDFs) and the web.
- Understand and contrast the two primary methods for tool usage in CrewAI: Agent-level versus Task-level assignment.
- Design more efficient, predictable, and maintainable agentic workflows by assigning tools directly to tasks.


---


## Setup


For this lab, we will be using the following libraries:

* [`crewai`](https://docs.crewai.com/introduction) for creating AI agents and orchestrating workflows.
* [`crewai_tools`](https://docs.crewai.com/tools/overview) to equip our agents and tasks with pre-built tools.
* [`PDFSearchTool`](https://docs.crewai.com/tools/file-document/pdfsearchtool) to search The Daily Dish FAQ PDF.
* [`TavilySearchTool`](https://docs.crewai.com/tools/search-research/tavilysearchtool) to search the web for current information.
* [`python-dotenv`](https://pypi.org/project/python-dotenv/) to load `OPENAI_API_KEY` and `TAVILY_API_KEY` from a `.env` file.
* [`crewai.tools.tool`](https://docs.crewai.com/tools/custom-tools) to create simple custom function tools.


### Installing Required Libraries

This project manages dependencies through `requirements.in` and the compiled environment files. Do **not** install packages from inside this notebook.

Before running the CrewAI cells, make sure your `.env` file contains:

```bash
OPENAI_API_KEY=your_openai_key
TAVILY_API_KEY=your_tavily_key
```


In [1]:
# Dependencies are managed outside this notebook.
# If an import fails, update requirements.in, then pip-compile and pip-sync manually.


### Importing Required Libraries


In [2]:
import os
import re
import warnings
from functools import reduce

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool
from crewai_tools import PDFSearchTool, TavilySearchTool
from dotenv import load_dotenv

warnings.filterwarnings("ignore")  # Keeps Jupyter Notebook clean (not part of functionality)
load_dotenv()

missing_keys = [key for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"] if not os.getenv(key)]
if missing_keys:
    print(f"Set these environment variables in your .env file before running CrewAI calls: {', '.join(missing_keys)}")


In [3]:
# No LiteLLM SSL override is required for the OpenAI/Tavily setup.


---


## Project Roadmap


### Agentic AI with CrewAI

At a high level, instead of designing complex, rigid code for this chatbot, we delegate the work to a team—or **Crew**—of AI Agents.

An **AI Agent** is an autonomous worker. Like any worker, it needs a clear job description, which we provide through specific parameters:
- **Role:** What is its primary function? (for example, "Customer Service Specialist")
- **Goal:** What is it trying to achieve? (for example, "Answer customer questions accurately")
- **Backstory:** What context does it need to perform its role effectively?
- **Tools:** What resources can it use to accomplish its goal? (for example, a PDF search tool, a web browser)

Once we have our agents, we define the **Tasks** they need to complete. A **Crew** then orchestrates this entire process, managing the agents and ensuring tasks are executed in the correct order to achieve the final objective.


### The Role of Tools: Agent-Centric versus Task-Centric

A key decision in CrewAI is *how* to give tools to your agents. There are two main strategies, and understanding the difference is crucial for building robust applications.

**1. Agent-Centric Approach (The Generalist):**
This is the most common method. You give an agent a 'toolbox' containing all the tools it might possibly need. The agent then uses its intelligence (the LLM's reasoning ability) to decide which tool is best for the situation at hand. This is flexible but can sometimes lead to the agent making mistakes or using tools inefficiently.

![Diagram 1](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/NPYpo2eWUERulpzrm9xYtw/Adobe%20Express%20-%20file.jpg)

**2. Task-Centric Approach (The Specialist):**
In this more advanced approach, you don't give the tools to the agent directly. Instead, you attach specific tools to the specific **Tasks** that require them. When the agent starts a task, it is temporarily granted access to only the tools needed for that job. This creates a much more focused and predictable workflow.

![Diagram 2.jpg](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/olP9UG8s4B7QnOJhCRH4aA/Diagram%202.jpg)


---


## Writing the Code


### Configuring our LLM

First, we configure the Language Model (LLM) that will power our agents' reasoning and language generation. This version uses OpenAI through CrewAI's current `LLM` interface. The API key is loaded from `.env` with `python-dotenv`, so credentials are not hard-coded in the notebook.


In [4]:
llm = LLM(model="openai/gpt-4o", temperature=0.2)


### Defining Tools for Information Gathering

Our chatbot needs to access information from two sources: Daily Dish's FAQ PDF and the general internet. To do this, we'll instantiate two tools.

1. **`PDFSearchTool`**: This tool, from the `crewai_tools` library, searches the provided PDF document. When used, it performs a semantic search to find the most relevant sections of the PDF related to a query.
2. **`TavilySearchTool`**: This tool performs web search through Tavily. It is useful when a customer asks for current information or the FAQ does not contain enough detail.

To use `TavilySearchTool`, set `TAVILY_API_KEY` in your `.env` file. The notebook loads it with `load_dotenv()`.


### Initializing Tavily Search Tool:


In [5]:
# TavilySearchTool reads TAVILY_API_KEY from the environment.
# Keep the key in .env; do not paste secrets into notebook cells.
print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))


Tavily key loaded: True


In [6]:
web_search_tool = TavilySearchTool()


### Creating our PDF Search Tool: 


In [8]:
#FAQ_PDF_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf"
FAQ_PDF_URL = "./data/The_Daily_Dish_FAQ.pdf"
pdf_search_tool = PDFSearchTool(pdf=FAQ_PDF_URL)


### Approach 1: The Standard Method (Agent-Centric Tools)

First, we'll build the chatbot using the conventional approach where we give our agent a toolbox with all the necessary tools. The agent will be responsible for deciding whether to search the PDF, use Tavily web search, or answer directly from its own reasoning.

#### **Step 1.1: Create the Agent**

We define an `Inquiry Specialist Agent` whose job is to answer questions. Notice that the `tools` parameter is a list containing the `pdf_search_tool` and `web_search_tool`.


In [9]:
agent_centric_agent = Agent(
    role="The Daily Dish Inquiry Specialist",
    goal=(
        "Accurately answer customer questions about The Daily Dish restaurant. "
        "Decide whether to use the restaurant FAQ PDF or Tavily web search."
    ),
    backstory=(
        "You are an AI assistant for The Daily Dish. You can search the official FAQ PDF "
        "for restaurant details and use Tavily web search when current external information is needed. "
        "Choose the most appropriate source for each customer question."
    ),
    tools=[pdf_search_tool, web_search_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm,
)


#### **Step 1.2: Define the Task**

We create a single, broad task that instructs the agent to handle the customer's query.


In [10]:
agent_centric_task = Task(
    description=(
        "Answer the following customer query: '{customer_query}'. "
        "Analyze the question and use the tools at your disposal, either PDF search or Tavily web search, "
        "to find the most relevant information. Synthesize the findings into a clear and friendly response."
    ),
    expected_output="A comprehensive and well-formatted answer to the customer's query.",
    agent=agent_centric_agent,
)


#### **Step 1.3: Assemble the Crew**

Finally, we create the Crew. It's a simple setup with our one agent and one task.


In [11]:
agent_centric_crew = Crew(
    agents=[agent_centric_agent],
    tasks=[agent_centric_task],
    process=Process.sequential,
    verbose=True,
)


The FAQ PDF is loaded directly from the URL in `FAQ_PDF_URL`, so no shell download step is required. If you prefer a local file, download it manually and set `PDFSearchTool(pdf="DailyDishFAQ.pdf")`.


In [12]:
# The FAQ PDF is referenced by URL in FAQ_PDF_URL.
print("FAQ source:", FAQ_PDF_URL)


FAQ source: ./data/The_Daily_Dish_FAQ.pdf


Try asking the following questions:

1. What are the timings?
2. What is the phone number?
3. What is the location?

You could also ask some combinations of other questions that you will find in the FAQ PDF itself.


In [13]:
sample_customer_query = "What are your phone number, hours, and parking options?"

result_agent_centric = agent_centric_crew.kickoff(
    inputs={"customer_query": sample_customer_query}
)

print("--- Agent-Centric Daily Dish Assistant ---")
print(result_agent_centric.raw)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 437eddc6-51d6-486d-a43a-c1616454e207                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following customer query: 'What are your phone number, hours, and parking options?'. Analyze  │
│  the question and use the tools at your disposal, either PDF search or Tavily web search, to find the most      │
│  relevant information. Synthesize the findings into a clear and friendly response.                              │
│  ID: 63b14142-3374-4d5a-9a98-3577b7caf81a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│  Task: Answer the following customer query: 'What are your phone number, hours, and parking options?'. Analyze  │
│  the question and use the tools at your disposal, either PDF search or Tavily web search, to find the most      │
│  relevant information. Synthesize the findings into a clear and friendly response.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Args: {'query': 'phone number, hours, parking options'}                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_a_pdfs_content executed with result: Relevant Content:
Page 1:

The Daily Dish - Frequently Asked Questions 

 

General Information & Location 

 

1.  Q: What are your hours of operation? 

    A: The Daily Dish is open Monday to Frida...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the information you requested about The Daily Dish:                                                    │
│                                                                                                                 │
│  - **Phone Number:** You can reach us at (555) 123-4567.                                                        │
│                                                                                                                 │
│  - **Hours of Operation:**                                                                                      │
│    - Monday to Friday: 11:00 AM to 10:00 PM                                                                     │
│    - Saturday and Sunday: 10:00 AM to 11:00 PM                                                                  │
│                                                                                                                 │
│  - **Parking Options:** We offer complimentary valet parking. There is also street parking available nearby.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the following customer query: 'What are your phone number, hours, and parking options?'. Analyze  │
│  the question and use the tools at your disposal, either PDF search or Tavily web search, to find the most      │
│  relevant information. Synthesize the findings into a clear and friendly response.                              │
│  Agent: The Daily Dish Inquiry Specialist                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 437eddc6-51d6-486d-a43a-c1616454e207                                                                       │
│  Final Output: Here is the information you requested about The Daily Dish:                                      │
│                                                                                                                 │
│  - **Phone Number:** You can reach us at (555) 123-4567.                                                        │
│                                                                                                                 │
│  - **Hours of Operation:**                                                                                      │
│    - Monday to Friday: 11:00 AM to 10:00 PM                                                                     │
│    - Saturday and Sunday: 10:00 AM to 11:00 PM                                                                  │
│                                                                                                                 │
│  - **Parking Options:** We offer complimentary valet parking. There is also street parking available nearby.    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- Agent-Centric Daily Dish Assistant ---
Here is the information you requested about The Daily Dish:

- **Phone Number:** You can reach us at (555) 123-4567.

- **Hours of Operation:** 
  - Monday to Friday: 11:00 AM to 10:00 PM
  - Saturday and Sunday: 10:00 AM to 11:00 PM

- **Parking Options:** We offer complimentary valet parking. There is also street parking available nearby.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

This approach works, and because we set `verbose=True` to Agent, we see the agent spending thought process on *which tool to choose*. For a simple query, this is fine. But in a complex workflow with many tools and steps, this ambiguity can lead to errors or inefficient tool usage. Now, let's see a better way.


### Approach 2: A More Focused Method (Task-Centric Tools)

Now, we'll refactor our solution to use a task-centric approach. We will create a multi-step process where each step (Task) has its own dedicated tool. This makes the agent's job simpler and the overall workflow more reliable.

Our new workflow will have three tasks:
1. **Search the FAQ:** This task will *only* use the `PDFSearchTool`.
2. **Search the web if useful:** This task will *only* use the `TavilySearchTool`.
3. **Draft the Response:** This task will use the information from the previous tasks to write the final answer. It needs no tools.

#### **Step 2.1: Create the Agent**

This time, we create a `Customer Service Specialist` agent. Notice the critical difference: the `tools` list is **empty**. We are not giving the agent its toolbox upfront. Instead, each task controls which tool is available.


In [14]:
task_centric_agent = Agent(
    role="Customer Service Specialist",
    goal="Provide exceptional customer service by following a multi-step process to answer customer questions accurately.",
    backstory=(
        "You are an AI assistant for The Daily Dish. You are excellent at following a structured workflow. "
        "For each task, use only the tool assigned to that task and pass useful findings to the next step."
    ),
    tools=[],  # The agent is not given any tools directly.
    verbose=True,
    allow_delegation=False,
    llm=llm,
)


#### **Step 2.2: Define the Tasks with Specific Tools**

Here is the core of the new approach. We define distinct tasks and use the `tools` parameter within the `Task` definition itself to assign a specific tool to that step.

- **`faq_search_task`** is exclusively paired with `pdf_search_tool`.
- **`web_context_task`** is exclusively paired with `web_search_tool`.
- **`response_drafting_task`** uses the output from both search tasks as context but requires no tools of its own.


In [15]:
faq_search_task = Task(
    description=(
        "Search the restaurant FAQ PDF for information related to this customer query: '{customer_query}'. "
        "Extract the most relevant official facts. If the FAQ does not contain the answer, say so clearly."
    ),
    expected_output="Relevant FAQ information, or a clear note that the FAQ did not contain the answer.",
    tools=[pdf_search_tool],  # Tool assigned directly to the task.
    agent=task_centric_agent,
)

web_context_task = Task(
    description=(
        "Search the web with Tavily for current public information related to this customer query: '{customer_query}'. "
        "Use this mainly for details that might not be in the FAQ, such as current local context or recent updates."
    ),
    expected_output="Relevant web search findings, or a clear note if web search was not needed.",
    tools=[web_search_tool],  # Different tool assigned to this task.
    agent=task_centric_agent,
)

response_drafting_task = Task(
    description=(
        "Using the FAQ findings and Tavily findings from the previous tasks, draft a friendly and comprehensive "
        "response to the customer's query: '{customer_query}'. Prefer official FAQ details when available."
    ),
    expected_output="The final, customer-facing response.",
    agent=task_centric_agent,
    context=[faq_search_task, web_context_task],
)


#### **Step 2.3: Assemble the New Crew**

We assemble our new crew, providing the single agent and the list of three tasks. The `Process.sequential` setting ensures the tasks run in the order we've listed them.


In [16]:
task_centric_crew = Crew(
    agents=[task_centric_agent],
    tasks=[faq_search_task, web_context_task, response_drafting_task],
    process=Process.sequential,
    verbose=True,
)


### Chatbot Execution

This final script runs one sample customer question. The core of the interaction is `task_centric_crew.kickoff(...)`. This method activates the task-centric system we just defined.

In the task-centric version, the agent no longer has to decide which tools are available. It executes Task 1 with the PDF tool, Task 2 with Tavily search, and Task 3 with the previous results as context. The process is more explicit and easier to debug.


Try asking the following questions:

1. What are the timings?
2. What is the phone number?
3. What is the location?

You could also ask some combinations of other questions that you will find in the FAQ PDF itself.


In [17]:
sample_customer_query = "What are your phone number, hours, and parking options?"

result_task_centric = task_centric_crew.kickoff(
    inputs={"customer_query": sample_customer_query}
)

print("--- Task-Centric Daily Dish Assistant ---")
print(result_task_centric.raw)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6cfbf522-b48e-4c67-b34b-614aa6fd362f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the restaurant FAQ PDF for information related to this customer query: 'What are your phone       │
│  number, hours, and parking options?'. Extract the most relevant official facts. If the FAQ does not contain    │
│  the answer, say so clearly.                                                                                    │
│  ID: ad198ab3-cc70-41a1-ad1f-2695a93dc064                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│  Task: Search the restaurant FAQ PDF for information related to this customer query: 'What are your phone       │
│  number, hours, and parking options?'. Extract the most relevant official facts. If the FAQ does not contain    │
│  the answer, say so clearly.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Args: {'query': 'phone number, hours, parking options'}                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_a_pdfs_content executed with result: Relevant Content:
Page 1:

The Daily Dish - Frequently Asked Questions 

 

General Information & Location 

 

1.  Q: What are your hours of operation? 

    A: The Daily Dish is open Monday to Frida...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello!                                                                                                         │
│                                                                                                                 │
│  Thank you for reaching out to us with your questions. I'm happy to provide you with the information you need.  │
│                                                                                                                 │
│  - **Phone Number:** You can contact us at (555) 123-4567.                                                      │
│                                                                                                                 │
│  - **Hours of Operation:** We are open Monday to Friday from 11:00 AM to 10:00 PM, and on Saturday and Sunday   │
│  from 10:00 AM to 11:00 PM.                                                                                     │
│                                                                                                                 │
│  - **Parking Options:** We offer complimentary valet parking for your convenience. Additionally, there is       │
│  street parking available nearby if you prefer.                                                                 │
│                                                                                                                 │
│  If you have any more questions or need further assistance, feel free to reach out. We look forward to          │
│  welcoming you soon!                                                                                            │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│  The Daily Dish Team                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the FAQ findings and Tavily findings from the previous tasks, draft a friendly and comprehensive   │
│  response to the customer's query: 'What are your phone number, hours, and parking options?'. Prefer official   │
│  FAQ details when available.                                                                                    │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6cfbf522-b48e-4c67-b34b-614aa6fd362f                                                                       │
│  Final Output: Hello!                                                                                           │
│                                                                                                                 │
│  Thank you for reaching out to us with your questions. I'm happy to provide you with the information you need.  │
│                                                                                                                 │
│  - **Phone Number:** You can contact us at (555) 123-4567.                                                      │
│                                                                                                                 │
│  - **Hours of Operation:** We are open Monday to Friday from 11:00 AM to 10:00 PM, and on Saturday and Sunday   │
│  from 10:00 AM to 11:00 PM.                                                                                     │
│                                                                                                                 │
│  - **Parking Options:** We offer complimentary valet parking for your convenience. Additionally, there is       │
│  street parking available nearby if you prefer.                                                                 │
│                                                                                                                 │
│  If you have any more questions or need further assistance, feel free to reach out. We look forward to          │
│  welcoming you soon!                                                                                            │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│  The Daily Dish Team                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- Task-Centric Daily Dish Assistant ---
Hello!

Thank you for reaching out to us with your questions. I'm happy to provide you with the information you need.

- **Phone Number:** You can contact us at (555) 123-4567.

- **Hours of Operation:** We are open Monday to Friday from 11:00 AM to 10:00 PM, and on Saturday and Sunday from 10:00 AM to 11:00 PM.

- **Parking Options:** We offer complimentary valet parking for your convenience. Additionally, there is street parking available nearby if you prefer.

If you have any more questions or need further assistance, feel free to reach out. We look forward to welcoming you soon!

Best regards,
The Daily Dish Team


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<!-- ## Conclusion

In this lab, you built a sophisticated customer service chatbot and, more importantly, explored a fundamental concept in CrewAI: the strategic assignment of tools.

You saw two approaches:
1.  **Agent-Centric:** Flexible and easy to set up, but relies on the agent's reasoning to select tools, which can be inefficient or unpredictable in complex scenarios.
2.  **Task-Centric:** More structured and robust. By assigning tools directly to the tasks that need them, you create a clear, deterministic, and efficient workflow.

**Key Advantages of Task-Centric Tools:**
- **Focus & Efficiency:** The agent doesn't waste time or processing power deciding which tool to use. It's told exactly what to use for each step.
- **Clarity & Maintainability:** The workflow is explicit and easy to follow. Anyone reading the code can see precisely which task uses which tool.
- **Control & Security:** This pattern allows for fine-grained control, granting an agent access to a powerful tool only for the specific duration of a single task.

Although the agent-centric approach has its place, mastering the task-centric method is a key step toward building truly professional, production-grade multi-agent systems with CrewAI.
 -->

## Conclusion

In this lab, you built a sophisticated customer service chatbot and, more importantly, explored a fundamental concept in CrewAI: the strategic assignment of tools.

You saw two approaches:
1. **Agent‑Centric:** Flexible and easy to set up, but relies on the agent's reasoning to select tools, which can be inefficient or unpredictable in complex scenarios.  
2. **Task‑Centric:** More structured and robust. By assigning tools directly to the tasks that need them, you create a clear, deterministic, and efficient workflow.

| Aspect             | Agent‑Centric Output                                                   | Task‑Centric Output                                                       |
|--------------------|-------------------------------------------------------------------------|----------------------------------------------------------------------------|
| **Predictability** | Varies run‑to‑run based on the LLM’s tool choice and phrasing.          | Consistent across runs thanks to fixed task‑to‑tool mapping.              |
| **Debuggability**  | Harder to trace—calls and errors are buried in the LLM’s reasoning.     | Easy to debug—each task’s inputs, outputs, and errors are explicitly logged. |
| **Reusability**    | You get a free‑form text blob that you must parse yourself.             | You get structured intermediate results (for example, JSON or code blocks) ready for reuse. |
| **Structure**¹     | Search and formatting are blended in one step.                          | A separate formatting task produces a clean, structured final message.    |

This “Structure” difference exists only because we introduced a dedicated formatting task in the Task‑Centric workflow.

**Key Advantages of Task‑Centric Tools:**
- **Focus & Efficiency:** The agent doesn't waste time or processing power deciding which tool to use. It's told exactly what to use for each step.  
- **Clarity & Maintainability:** The workflow is explicit and easy to follow. Anyone reading the code can see precisely which task uses which tool.  
- **Control & Security:** This pattern allows for fine‑grained control, granting an agent access to a powerful tool only for the specific duration of a single task.

While the agent‑centric approach has its place, mastering the task‑centric method is a key step toward building truly professional, production‑grade multi‑agent systems with CrewAI.


## Extending CrewAI with Custom Functions


In CrewAI, **tools** (or Functions) are functional components that agents can call to perform specific actions, such as searching the web, reading PDFs, or doing calculations. Although CrewAI provides several built-in tools, we can also create our **own custom tools** to extend its capabilities for domain-specific or utility tasks.

Custom tools are simply Python functions that are **wrapped using the `@tool` decorator** from `crewai.tools`. This allows CrewAI agents to recognize, reason about, and invoke those functions during task execution.


Let's see how we can create an **add tool** using CrewAI’s `@tool` decorator. Just like LangChain allows custom tool creation using its own `@tool` decorator, CrewAI also provides a simple way to register functions as tools that agents can invoke during task execution.

To define a tool in CrewAI:

- You annotate your function with `@tool("Tool Name")`.
- You provide a **docstring**, which acts as the tool's self-description and helps the language model understand what the tool does.
- Then you implement the **function body**, which contains the actual logic.


In [18]:
@tool("Add Two Numbers Tool")
def add_numbers(data: str) -> int:
    """
    Extracts and adds integers from the input string.
    Example input: 'add 1 and 2' or '[1,2,3,4]'
    Output: sum of the numbers
    """
    # Find all integers in the string.
    numbers = list(map(int, re.findall(r"-?\d+", data)))
    return sum(numbers)


Now, let's create another tool called `multiply_numbers`. This tool takes a string input such as `"multiply 2 and 3"` or `"values are [2,3,4]"`, extracts all the integers from the text, and returns their **product** as an integer.


In [19]:
@tool("Multiply Numbers Tool")
def multiply_numbers(data: str) -> int:
    """
    Extracts and multiplies integers from the input string.
    Example input: 'multiply 2 and 3' or '[2,3,4]'
    Output: the product of all numbers found
    """
    numbers = list(map(int, re.findall(r"-?\d+", data)))
    return reduce(lambda x, y: x * y, numbers, 1)


Now, we create a `Calculator Agent` with a clear goal being to extract, add or multiply numbers based on a user's query. We provide the relevant backstory, the tools that we created, and the LLM itself.


In [20]:
calculator_agent = Agent(
    role="Calculator",
    goal="Extracts, adds, or multiplies numbers when asked, using the Add Two Numbers and Multiply Numbers tools.",
    backstory="An expert at parsing numeric instructions and computing sums or products.",
    tools=[add_numbers, multiply_numbers],
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


We also create a `Calculation Task` by providing a clear description, an expected output, and an agent.


In [21]:
calculation_task = Task(
    description="Extract numbers from '{numbers}' and either add or multiply them, depending on the natural-language instruction.",
    expected_output="An integer result, either sum or product, based on the user's request.",
    agent=calculator_agent,
)


Now let's bring together the created agent and task in a `Crew`.


In [22]:
calculator_crew = Crew(
    agents=[calculator_agent],
    tasks=[calculation_task],
    process=Process.sequential,
    verbose=True,
)


Let's run the crew by providing a user query in `.kickoff()` and check the output.


In [23]:
# Inputs for addition.
sum_result = calculator_crew.kickoff(inputs={"numbers": "please add 4, 5, and 6"})
print("Sum result:", sum_result.raw)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5000dd7a-080f-43c6-871b-15b31e4dc443                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Extract numbers from 'please add 4, 5, and 6' and either add or multiply them, depending on the          │
│  natural-language instruction.                                                                                  │
│  ID: 08515708-afa8-4084-8c98-4da54819c5a4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Calculator                                                                                              │
│                                                                                                                 │
│  Task: Extract numbers from 'please add 4, 5, and 6' and either add or multiply them, depending on the          │
│  natural-language instruction.                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool add_two_numbers_tool executed with result: 15...

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: add_two_numbers_tool                                                                                     │
│  Args: {'data': 'please add 4, 5, and 6'}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Calculator                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  15                                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Extract numbers from 'please add 4, 5, and 6' and either add or multiply them, depending on the          │
│  natural-language instruction.                                                                                  │
│  Agent: Calculator                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5000dd7a-080f-43c6-871b-15b31e4dc443                                                                       │
│  Final Output: 15                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sum result: 15


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Inputs for multiplication.
product_result = calculator_crew.kickoff(inputs={"numbers": "multiply 7 and 8 also 9 dont forget 10"})
print("Product result:", product_result.raw)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5000dd7a-080f-43c6-871b-15b31e4dc443                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Extract numbers from 'multiply 7 and 8 also 9 dont forget 10' and either add or multiply them,           │
│  depending on the natural-language instruction.                                                                 │
│  ID: 08515708-afa8-4084-8c98-4da54819c5a4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Calculator                                                                                              │
│                                                                                                                 │
│  Task: Extract numbers from 'multiply 7 and 8 also 9 dont forget 10' and either add or multiply them,           │
│  depending on the natural-language instruction.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Authors

[Abdul Fatir](https://www.linkedin.com/in/abdul-fatir)

Abdul specializes in Data Science, Machine Learning, and AI. He has deep expertise in understanding how the latest technologies work, and their applications.  
Feel free to contact him with questions about this project or any other AI/ML topics.

[Karan Goswami](https://author.skills.network/instructors/karan_goswami) is a Data Scientist at IBM and is pursuing Master's student at McMaster university with a major in AI. He is fluent in Generative AI, AI/ML topics. He has many projects published on [CognitiveClass.ai](https://cognitiveclass.ai). Feel free to connect with him if you need any help or just want to connect!


## Change Log

<details>
    <summary>Click here for the changelog</summary>

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2025-07-27|1.0|Abdul Fatir|Initial version created|
|2025-08-02|1.1|Steve Ryan|ID review and format fixes|
|2025-08-04|1.2|Leah Hanson|QA review and grammar/IBM style guide adherencefixes|
</details>

---



Copyright © IBM Corporation. All rights reserved.
